# ViT5 inference trên 500 bài Vietnews test

Settings: GPU T4 x2, **Internet On**, notebook Private. **Không cần upload dataset.**
Cell lấy file sẽ clone/tải `data/test_tokenized` từ GitHub (`ThanhChinhBK/vietnews`), rồi lấy đúng `000001.txt.seg` … `000500.txt.seg`.
Model: `VietAI/vit5-base-vietnews-summarization`

**Bắt buộc:** cell 1 gỡ transformers 5 rồi cài 4.44.2. Xong thì **Restart session**, chạy từ cell import. Đừng Run All.

In [ ]:
!nvidia-smi
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" sentencepiece protobuf
import transformers, tokenizers
print("transformers", transformers.__version__, "tokenizers", tokenizers.__version__)

Sau cell trên: **Restart session**. In ra `transformers 4.44.2` mới chạy tiếp. Nếu vẫn 4.5x thì restart chưa xong.

In [ ]:
from pathlib import Path
import json
import subprocess
import urllib.request
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.44"), "Restart session sau pip, rồi chạy lại cell này"

MODEL = "VietAI/vit5-base-vietnews-summarization"
IDS = [f"{i:06d}.txt.seg" for i in range(1, 501)]

def existing_files():
    found = {}
    for p in Path("/kaggle/input").rglob("*.txt.seg"):
        found[p.name] = p
    for p in Path("/kaggle/working").rglob("*.txt.seg"):
        found.setdefault(p.name, p)
    files = [found[name] for name in IDS if name in found]
    return files

def fetch_from_github():
    dest = Path("/kaggle/working/test_tokenized")
    dest.mkdir(parents=True, exist_ok=True)
    repo = Path("/kaggle/working/vietnews")
    test_dir = repo / "data" / "test_tokenized"
    try:
        if not test_dir.exists():
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                "https://github.com/ThanhChinhBK/vietnews.git", str(repo),
            ])
            subprocess.check_call(["git", "-C", str(repo), "sparse-checkout", "set", "data/test_tokenized"])
        if (test_dir / IDS[0]).exists():
            print("github git", test_dir)
            return [test_dir / name for name in IDS]
    except Exception as e:
        print("git sparse thất bại, tải 500 file raw:", e)
    base = "https://raw.githubusercontent.com/ThanhChinhBK/vietnews/master/data/test_tokenized"
    for i, name in enumerate(IDS, 1):
        out = dest / name
        if not out.exists() or out.stat().st_size == 0:
            urllib.request.urlretrieve(f"{base}/{name}", out)
        if i % 50 == 0:
            print("downloaded", i)
    return [dest / name for name in IDS]

files = existing_files()
if len(files) < 500:
    print("input chưa đủ 500, lấy từ GitHub. hiện có", len(files))
    files = fetch_from_github()
missing = [p.name for p in files if not p.exists()]
assert not missing, missing[:5]
assert files[0].name == "000001.txt.seg" and files[-1].name == "000500.txt.seg"
print("n files", len(files))
print("first", files[0])
print("last", files[-1])

In [ ]:
def parse_file(path):
    parts = [p.strip() for p in Path(path).read_text(encoding="utf-8").split("\n\n") if p.strip()]
    return {"id": Path(path).name, "title": parts[0], "abstract": parts[1], "body": "\n".join(parts[2:])}

docs = [parse_file(p) for p in files]
print(docs[0]["id"], len(docs[0]["body"].split()), "n=", len(docs))

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device)

tok = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to(device)
model.eval()

sample = docs[0]["body"] + "</s>"
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
enc = {k: v.to(device) for k, v in enc.items()}
with torch.no_grad():
    out = model.generate(**enc, max_length=256, early_stopping=True)
print(tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))

In [ ]:
from pathlib import Path
import json

assert "docs" in dir() and "tok" in dir() and "model" in dir(), (
    "Chạy tuần tự: cell import → parse docs → load model → cell này. Không chạy cell này một mình."
)

out_path = Path("/kaggle/working/preds_500.json")
preds = json.loads(out_path.read_text(encoding="utf-8")) if out_path.exists() else []
done = {row["id"] for row in preds}
print("resume", len(done))
for i, doc in enumerate(docs):
    if doc["id"] in done:
        continue
    text = doc["body"] + "</s>"
    enc = tok(text, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_length=256, early_stopping=True)
    pred = tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    preds.append({"id": doc["id"], "abstract": doc["abstract"], "pred": pred})
    done.add(doc["id"])
    if len(preds) % 25 == 0:
        out_path.write_text(json.dumps(preds, ensure_ascii=False), encoding="utf-8")
        print("saved", len(preds))
out_path.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out_path, "n=", len(preds))